In [ ]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Literal
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field
import os


In [ ]:
load_dotenv()
api_key = os.getenv('GOOGLE_API_KEY')
model_name = os.getenv('GEMINI_MODEL', 'gemini-2.5-flash')

llm = ChatGoogleGenerativeAI(
    model=model_name,
    api_key=api_key,
)


In [ ]:
class TweetState(TypedDict):
    topic: str
    tweet: str
    evaluation: Literal['approved', 'needs_improvement']
    feedback: str
    iteration: int
    max_iteration: int


In [ ]:
class TweetEvaluation(BaseModel):
    evaluation: Literal['approved', 'needs_improvement'] = Field(
        ..., description='Final evaluation of the tweet'
    )
    feedback: str = Field(
        ..., description='Constructive feedback to improve the tweet'
    )


In [ ]:
structured_evaluator = llm.with_structured_output(TweetEvaluation)


In [ ]:
def generate_tweet(state: TweetState) -> dict:
    '''First attempt at the tweet.'''
    messages = [
        SystemMessage(content='You are a funny and clever Twitter/X influencer.'),
        HumanMessage(content=(
            f"Write a short, original, and hilarious tweet on the topic: {state['topic']}.
"
            'Rules:
'
            '- Do not use question-answer format
'
            '- Max 280 characters
'
            '- Use observational humor, irony, sarcasm, or cultural references'
        )),
    ]
    response = llm.invoke(messages).content
    return {'tweet': response, 'iteration': 1, 'max_iteration': 3}


In [ ]:
def evaluate_tweet(state: TweetState) -> dict:
    '''Structured evaluation of the tweet.'''
    messages = [
        SystemMessage(content=(
            'You are an impartial tweet evaluator. Judge tweets on humor, '
            'originality and clarity. Respond with your structured decision.'
        )),
        HumanMessage(content=(
            f"Evaluate the tweet on topic '{state['topic']}':
"
            f"{state['tweet']}"
        )),
    ]
    evaluated = structured_evaluator.invoke(messages)
    return {'evaluation': evaluated.evaluation, 'feedback': evaluated.feedback}


In [ ]:
def optimize_tweet(state: TweetState) -> dict:
    '''Rewrite the tweet using the feedback.'''
    messages = [
        SystemMessage(content=(
            'You are a witty Twitter/X influencer who rewrites drafts using feedback.'
        )),
        HumanMessage(content=(
            f"Rewrite this tweet on the topic '{state['topic']}' applying the feedback.
"
            f"Original tweet:
{state['tweet']}
"
            f"Feedback:
{state['feedback']}"
        )),
    ]
    response = llm.invoke(messages).content
    return {'tweet': response, 'iteration': state['iteration'] + 1}


In [ ]:
def route_evaluation(state: TweetState) -> Literal['approved', 'needs_improvement']:
    if state['evaluation'] == 'approved':
        return 'approved'
    if state['iteration'] >= state['max_iteration']:
        return 'approved'
    return 'needs_improvement'


In [ ]:
graph = StateGraph(TweetState)

graph.add_node('generate', generate_tweet)
graph.add_node('evaluate', evaluate_tweet)
graph.add_node('optimize', optimize_tweet)

graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')
graph.add_conditional_edges('evaluate', route_evaluation, {
    'approved': END,
    'needs_improvement': 'optimize',
})
graph.add_edge('optimize', 'evaluate')

workflow = graph.compile()


In [ ]:
result = workflow.invoke({'topic': 'Indian trains'})
print('Iterations:', result['iteration'])
print(result['tweet'])
